# CPU Benchmark: Energy Conservation in Integrators

This notebook benchmarks all three integration schemes for the Hénon-Heiles system and compares their energy conservation properties.

## Key Questions:
- How does energy drift differ across methods?
- Can we recover the chaotic structure despite numerical errors?
- What makes symplectic integration special?

In [ ]:
import sys
sys.path.insert(0, '../src/cpu')

import numpy as np
import matplotlib.pyplot as plt
from integrators import (
    EulerIntegrator, RK4Integrator, SymplecticIntegrator,
    batch_integrate, create_initial_condition
)
from analysis import EnergyAnalysis, setup_output_dir

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## Configuration

In [ ]:
# Simulation parameters
n_trajectories = 100
n_steps = 10000
dt = 0.001
total_time = n_steps * dt

print(f"Configuration:")
print(f"  Trajectories: {n_trajectories}")
print(f"  Steps: {n_steps}")
print(f"  Timestep: {dt}")
print(f"  Total time: {total_time:.2f}")

## Generate Initial Conditions

In [ ]:
# Create initial ensemble in chaotic regime
initial_states = create_initial_condition(n_trajectories, seed=42)
print(f"Created {n_trajectories} random trajectories in chaotic regime")

## Run Benchmarks

This will run all three integrators and compare their energy conservation.

In [ ]:
import time

analysis = EnergyAnalysis()

integrators_list = [
    ('Euler', EulerIntegrator),
    ('RK4', RK4Integrator),
    ('Symplectic', SymplecticIntegrator)
]

results = {}

for method_name, integrator_class in integrators_list:
    print(f"\nRunning {method_name}...")
    
    # Deep copy initial states
    states = [s.copy() for s in initial_states]
    
    # Time the integration
    t_start = time.time()
    final_states, integrators_run = batch_integrate(
        states, dt, n_steps, integrator_class
    )
    t_elapsed = time.time() - t_start
    
    # Collect energy history
    energies_history = []
    for integrator in integrators_run:
        energies_history.append(np.array(integrator.energies).T)
    
    energies_all = np.concatenate(energies_history, axis=1)
    times = np.array(integrators_run[0].times)
    
    analysis.add_result(method_name.lower(), energies_all, times)
    
    # Print statistics
    E_initial = np.mean(energies_all[0, :])
    E_final = np.mean(energies_all[-1, :])
    dE_max = np.max(np.abs(energies_all - E_initial))
    
    results[method_name] = {
        'time': t_elapsed,
        'E_initial': E_initial,
        'E_final': E_final,
        'dE_max': dE_max
    }
    
    print(f"  Time: {t_elapsed:.3f}s")
    print(f"  Initial E: {E_initial:.6f}")
    print(f"  Max drift: {dE_max:.2e}")

## Energy Conservation Analysis

This is the key result: showing how well each method preserves energy over time.

In [ ]:
# Print comprehensive analysis
analysis.print_summary()

## Visualization

In [ ]:
# Plot energy drift comparison
fig, axes = analysis.plot_energy_drift()
plt.show()

## Summary Table

In [ ]:
import pandas as pd

# Create summary dataframe
summary_data = []
for method_name, result in results.items():
    summary_data.append({
        'Method': method_name,
        'Time (s)': f"{result['time']:.3f}",
        'Initial Energy': f"{result['E_initial']:.6f}",
        'Max Drift': f"{result['dE_max']:.2e}",
    })

df = pd.DataFrame(summary_data)
print("\nMethod Comparison:")
print(df.to_string(index=False))

## Conclusion

The symplectic integrator dramatically outperforms the naive Euler method and maintains
significantly better energy conservation than RK4 over long integration times.

This demonstrates that **structure preservation** is not just theoretical—it's essential
for accurate long-term Hamiltonian simulation.